In [28]:
# Đoạn code này tự động lấy Tên tài khoản và Tên máy tính
import getpass
import socket
user = getpass.getuser()
host = socket.gethostname()
def get_system_context():
    user = getpass.getuser()
    host = socket.gethostname()
    return f"[USER: {user}] [HOST: {host}]"

In [29]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [30]:
listings_df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .load("hdfs://master-node:9000/AirbnbData/Listings.csv")
listings_df_raw.show(truncate=False)

+----------+---------------------------------------------------+--------+----------+----------------------------+------------------+------------------+--------------------+-----------------+-------------------------+--------------------+----------------------+-------------------+--------+-----+--------+---------+----------------+------------+------------+--------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+--------------+--------------+--------------------+----------------------+-------------------------+---------------------+---------------------------+----------------------+-------------------+----------------+
|listing_id|na

In [32]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import ArrayType, StringType
listings_df = listings_df_raw.withColumn(
    "amenities",
    from_json(col("amenities"), ArrayType(StringType()))
)

In [33]:
from pyspark.sql.functions import when, col
# Quy đổi toàn bộ giá từ tiền tệ địa phương (Local Currency) sang USD
# Tỷ giá được cập nhật theo thời điểm hiện tại
listings_df_currency = listings_df.withColumn(
    "price_usd",
    when(col("city") == "New York", col("price") * 1.0)           # USD giữ nguyên
    .when(col("city") == "Paris", col("price") * 1.156)            # EUR -> USD
    .when(col("city") == "Rome", col("price") * 1.156)             # EUR -> USD
    .when(col("city") == "Sydney", col("price") * 0.7038)           # AUD -> USD
    .when(col("city") == "Hong Kong", col("price") * 0.1276)        # HKD -> USD
    .when(col("city") == "Bangkok", col("price") * 0.03048)         # THB -> USD
    .when(col("city") == "Mexico City", col("price") * 0.05795)     # MXN -> USD
    .when(col("city") == "Cape Town", col("price") * 0.06130)       # ZAR -> USD
    .when(col("city") == "Istanbul", col("price") *  0.02162)        # TRY -> USD
    .when(col("city") == "Rio de Janeiro", col("price") * 0.1961 )   # BRL -> USD
    .otherwise(col("price"))
)

In [34]:
listings_df_currency.show()

+----------+--------------------+--------+----------+--------------------+------------------+------------------+--------------------+-----------------+-------------------------+--------------------+----------------------+-------------------+--------+-----+--------+---------+----------------+------------+------------+--------+--------------------+-----+--------------+--------------+--------------------+----------------------+-------------------------+---------------------+---------------------------+----------------------+-------------------+----------------+------------------+
|listing_id|                name| host_id|host_since|       host_location|host_response_time|host_response_rate|host_acceptance_rate|host_is_superhost|host_total_listings_count|host_has_profile_pic|host_identity_verified|      neighbourhood|district| city|latitude|longitude|   property_type|   room_type|accommodates|bedrooms|           amenities|price|minimum_nights|maximum_nights|review_scores_rating|review_scor

In [35]:
listings_df_currency.printSchema()

root
 |-- listing_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- host_id: integer (nullable = true)
 |-- host_since: date (nullable = true)
 |-- host_location: string (nullable = true)
 |-- host_response_time: string (nullable = true)
 |-- host_response_rate: double (nullable = true)
 |-- host_acceptance_rate: double (nullable = true)
 |-- host_is_superhost: string (nullable = true)
 |-- host_total_listings_count: integer (nullable = true)
 |-- host_has_profile_pic: string (nullable = true)
 |-- host_identity_verified: string (nullable = true)
 |-- neighbourhood: string (nullable = true)
 |-- district: string (nullable = true)
 |-- city: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- property_type: string (nullable = true)
 |-- room_type: string (nullable = true)
 |-- accommodates: integer (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- amenities: array (nullable = true)
 |    |-- el

In [36]:
reviews_df=spark.read.csv("hdfs://master-node:9000/AirbnbData/Reviews.csv",header=True,inferSchema=True)
reviews_df.show()

+----------+---------+----------+-----------+
|listing_id|review_id|      date|reviewer_id|
+----------+---------+----------+-----------+
|     11798|330265172|2018-09-30|   11863072|
|     15383|330103585|2018-09-30|   39147453|
|     16455|329985788|2018-09-30|    1125378|
|     17919|330016899|2018-09-30|  172717984|
|     26827|329995638|2018-09-30|   17542859|
|     74561|330089224|2018-09-30|  173044789|
|    140355|330194958|2018-09-30|  160093807|
|    162163|329980859|2018-09-30|   94026758|
|    167998|329950677|2018-09-30|   35388162|
|    178188|330213008|2018-09-30|    3652511|
|    232956|330140226|2018-09-30|   57664145|
|    266725|330038354|2018-09-30|   77387165|
|    314288|330299075|2018-09-30|  192717587|
|    325880|330081449|2018-09-30|  154527568|
|    335003|329968377|2018-09-30|    3461699|
|    348747|330131287|2018-09-30|    9554201|
|    352851|330201364|2018-09-30|  142182690|
|    378714|330246144|2018-09-30|   15772951|
|    406852|330283854|2018-09-30| 

In [37]:
reviews_df.printSchema()

root
 |-- listing_id: integer (nullable = true)
 |-- review_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- reviewer_id: integer (nullable = true)



In [38]:
listings_df_currency.createOrReplaceTempView("listings")
reviews_df.createOrReplaceTempView("reviews")
spark.catalog.listTables()

[Table(name='listings', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='reviews', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [39]:
# Minh chứng số dòng > 100.000 dòng
spark.sql("SELECT COUNT(*) AS Total_rows FROM listings").show()

+----------+
|Total_rows|
+----------+
|    279712|
+----------+



In [40]:
# Minh chứng số cột >10
num_cols = len(listings_df.columns)
print(f"Tổng số cột: {num_cols}")

Tổng số cột: 33
